# 02 — Entrenamiento del Modelo

Clasificación multi-label de patologías en radiografías de tórax.
**Modelo:** DenseNet121 preentrenado en ImageNet
**Hardware:** Google Colab GPU (T4 / A100)

Este notebook cubre:
- **(a)** Modelo preentrenado y estrategia de fine-tuning
- **(b)** Configuración del entrenamiento
- **(c)** 3 experimentos comparativos (estrategias de incertidumbre CheXpert)
- **(d)** Evaluación final en test
- **(e)** Guardado del modelo

## Setup

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Ajustar esta ruta a donde subiste la carpeta del dataset en Drive
    DATASET_ROOT = "/content/drive/MyDrive/dataset"
else:
    DATASET_ROOT = os.path.abspath(os.path.join(".."))

DATA_DIR = os.path.join(DATASET_ROOT, "data")
DEV_DIR  = os.path.join(DATASET_ROOT, "dev")

print(f"DATASET_ROOT : {DATASET_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
assert os.path.exists(DATA_DIR), f"No se encuentra data/: {DATA_DIR}"

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "-q", "scikit-learn"], check=True)
    print("Dependencias instaladas.")

In [ ]:
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"Device      : {DEVICE}")

In [ ]:
CANONICAL_LABELS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Pleural_Effusion", "Pneumonia", "Pneumothorax",
    "Infiltration", "Nodule_Mass", "Pleural_Thickening", "No_Finding",
]
NUM_CLASSES = len(CANONICAL_LABELS)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

BATCH_SIZE   = 64
NUM_WORKERS  = 4 if IN_COLAB else 0
LR_HEAD      = 1e-3
LR_FULL      = 1e-4
EPOCHS_P1    = 3    # fase 1: solo head congelado
EPOCHS_P2    = 12   # fase 2: full fine-tune
WEIGHT_DECAY = 1e-5

print(f"Clases: {NUM_CLASSES}  |  Batch: {BATCH_SIZE}  |  Workers: {NUM_WORKERS}")

## Data

Se reutiliza `ChestXRayDataset` del notebook anterior.
La diferencia clave es el parámetro `nan_fill`:

| Experimento | `nan_fill` | Comportamiento |
|---|---|---|
| U-Ignore | `None` | NaN permanece → máscara en la loss |
| U-Zeros  | `0.0`  | NaN → 0 (negativo) |
| Label Smoothing | `0.55` | NaN → 0.55 (incierto como levemente positivo) |

In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self, csv_path, root_dir, labels=CANONICAL_LABELS,
                 transform=None, nan_fill=None):
        self.df       = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.labels   = labels
        self.transform = transform
        self.nan_fill  = nan_fill

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        img      = Image.open(img_path).convert("RGB")

        vals = row[self.labels].values.astype("float32")
        if self.nan_fill is not None:
            vals = np.where(np.isnan(vals), self.nan_fill, vals)
        labels = torch.tensor(vals)

        if self.transform:
            img = self.transform(img)
        return img, labels

print("ChestXRayDataset definida.")

In [ ]:
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms definidos.")

## (a) Modelo preentrenado y fine-tuning

**DenseNet121** — misma arquitectura del paper CheXNet (Rajpurkar et al., 2017) que entrenó sobre ChestXray8.

**Modificación:** reemplazar el clasificador final `(1024 → 1000 clases ImageNet)` por `(1024 → 11 clases)`.

**Estrategia de fine-tuning en 2 fases:**

| Fase | Épocas | Capas entrenadas | LR | Justificación |
|---|---|---|---|---|
| 1 — Warm-up | 3 | Solo `model.classifier` (head) | 1e-3 | Adaptar el head sin destruir los features ImageNet |
| 2 — Full fine-tune | 12 | Todas las capas | 1e-4 | Dataset grande (265k) justifica ajustar capas profundas; imágenes médicas ≠ ImageNet |

El fine-tuning completo es viable con este tamaño de dataset. Con datasets pequeños (<10k imágenes), congelar el backbone sería más seguro para evitar overfitting.

In [ ]:
def build_densenet121(num_classes=NUM_CLASSES, pretrained=True):
    weights = "IMAGENET1K_V1" if pretrained else None
    model = models.densenet121(weights=weights)
    # Reemplazar clasificador: 1024 -> num_classes
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model

def freeze_backbone(model):
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

model_test = build_densenet121()
print(f"DenseNet121 creado.")
print(f"  Parámetros totales   : {sum(p.numel() for p in model_test.parameters()):,}")
print(f"  Parámetros head      : {sum(p.numel() for p in model_test.classifier.parameters()):,}")
del model_test

## Funciones de pérdida

### U-Ignore — Masked BCE
Los labels `NaN` se omiten del cálculo de la loss usando una máscara binaria.
Ventaja: no introduce ruido. Desventaja: menos señal de entrenamiento.

### U-Zeros y Label Smoothing — BCE estándar
Los NaN se reemplazan antes de llegar a la loss (en el Dataset), por lo que se usa `BCEWithLogitsLoss` estándar.

### Justificación del Label Smoothing (Yuan et al., ICCV 2021)
El paper propone asignar `ε = 0.55` a los labels inciertos (`-1` en CheXpert).
Razón clínica: un hallazgo "incierto" tiene mayor probabilidad de ser verdadero positivo que verdadero negativo.
El valor 0.55 sesga ligeramente hacia positivo sin introducir el ruido fuerte de U-Ones (ε=1).

In [ ]:
def masked_bce(logits, targets):
    \"\"\"BCE ignorando posiciones NaN (estrategia U-Ignore).\"\"\"
    mask = ~torch.isnan(targets)
    t = targets.clone()
    t[~mask] = 0.0
    loss = F.binary_cross_entropy_with_logits(logits, t, reduction="none")
    denom = mask.float().sum().clamp(min=1)
    return (loss * mask.float()).sum() / denom

def standard_bce(logits, targets):
    \"\"\"BCE estándar (para U-Zeros y Label Smoothing, NaN ya reemplazado en Dataset).\"\"\"
    return F.binary_cross_entropy_with_logits(logits, targets)

print("Funciones de pérdida definidas.")

## Loop de entrenamiento

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits).sigmoid().numpy()
    all_labels = torch.cat(all_labels).numpy()

    # AUC-ROC por clase (ignora columnas con solo un valor único o NaN)
    aucs = []
    for i in range(all_labels.shape[1]):
        col = all_labels[:, i]
        valid = ~np.isnan(col)
        if valid.sum() > 0 and len(np.unique(col[valid])) > 1:
            aucs.append(roc_auc_score(col[valid], all_logits[valid, i]))
        else:
            aucs.append(float("nan"))

    mean_auc = np.nanmean(aucs)
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, mean_auc, np.array(aucs)


def run_experiment(exp_name, nan_fill, criterion, save_path, device,
                   epochs_p1=EPOCHS_P1, epochs_p2=EPOCHS_P2):
    print(f"\n{'='*55}")
    print(f"  {exp_name}")
    print(f"{'='*55}")

    # DataLoaders con la política de NaN elegida
    tr_ds  = ChestXRayDataset(os.path.join(DATA_DIR, "train.csv"),
                               DATASET_ROOT, transform=train_transform, nan_fill=nan_fill)
    val_ds = ChestXRayDataset(os.path.join(DATA_DIR, "val.csv"),
                               DATASET_ROOT, transform=val_test_transform, nan_fill=nan_fill)
    tr_loader  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=device.type=="cuda")
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=device.type=="cuda")

    model = build_densenet121().to(device)

    history = {"train_loss": [], "val_loss": [], "val_auc": []}

    # --- Fase 1: solo head ---
    print(f"\nFase 1 — Warm-up ({epochs_p1} épocas, solo head, LR={LR_HEAD})")
    freeze_backbone(model)
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY
    )
    for epoch in range(1, epochs_p1 + 1):
        tl = train_one_epoch(model, tr_loader, opt, criterion, device)
        vl, vauc, _ = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_auc"].append(vauc)
        print(f"  Epoch {epoch:2d}/{epochs_p1} — loss_train={tl:.4f}  loss_val={vl:.4f}  AUC_val={vauc:.4f}")

    # --- Fase 2: full fine-tune ---
    print(f"\nFase 2 — Full fine-tune ({epochs_p2} épocas, LR={LR_FULL})")
    unfreeze_all(model)
    opt = torch.optim.AdamW(model.parameters(), lr=LR_FULL, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs_p2)
    best_auc, best_state = 0.0, None
    for epoch in range(1, epochs_p2 + 1):
        tl = train_one_epoch(model, tr_loader, opt, criterion, device)
        vl, vauc, per_class = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_auc"].append(vauc)
        print(f"  Epoch {epoch:2d}/{epochs_p2} — loss_train={tl:.4f}  loss_val={vl:.4f}  AUC_val={vauc:.4f}")
        if vauc > best_auc:
            best_auc = vauc
            best_state = copy.deepcopy(model.state_dict())

    # Guardar mejor checkpoint
    torch.save(best_state, save_path)
    print(f"\nMejor AUC val: {best_auc:.4f} — guardado en {save_path}")

    return history, best_auc, per_class

print("Funciones de entrenamiento definidas.")

## (c) Experimento 1 — U-Ignore (Masked BCE, Baseline)

**Política de incertidumbre:** los labels NaN se omiten del cálculo de la loss.
Esta es la estrategia conservadora: no asumimos nada sobre los labels inciertos.

In [ ]:
hist_e1, auc_e1, per_class_e1 = run_experiment(
    exp_name  = "Exp 1 — U-Ignore (Masked BCE)",
    nan_fill  = None,       # NaN permanece → masked_bce lo omite
    criterion = masked_bce,
    save_path = os.path.join(DEV_DIR, "exp1_densenet121.pth"),
    device    = DEVICE,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
total_epochs = EPOCHS_P1 + EPOCHS_P2
x = range(1, total_epochs + 1)

ax1.plot(x, hist_e1["train_loss"], label="Train")
ax1.plot(x, hist_e1["val_loss"],   label="Val")
ax1.axvline(EPOCHS_P1 + 0.5, color="gray", linestyle="--", alpha=0.5, label="Fase 1→2")
ax1.set_title("Exp 1 — Loss")
ax1.set_xlabel("Época")
ax1.legend()

ax2.plot(x, hist_e1["val_auc"], color="darkorange")
ax2.axvline(EPOCHS_P1 + 0.5, color="gray", linestyle="--", alpha=0.5)
ax2.set_title("Exp 1 — AUC-ROC Val")
ax2.set_xlabel("Época")
ax2.set_ylim(0, 1)

plt.suptitle("Experimento 1: U-Ignore (Masked BCE)", fontsize=12)
plt.tight_layout()
plt.show()

## (c) Experimento 2 — U-Zeros (BCE estándar)

**Política de incertidumbre:** los labels inciertos (-1 en CheXpert, guardados como NaN) se mapean a **0** (negativo).
Más ejemplos disponibles para la loss, pero introduce falsos negativos para los casos inciertos.

In [ ]:
hist_e2, auc_e2, per_class_e2 = run_experiment(
    exp_name  = "Exp 2 — U-Zeros (BCE estándar)",
    nan_fill  = 0.0,         # NaN → 0
    criterion = standard_bce,
    save_path = os.path.join(DEV_DIR, "exp2_densenet121.pth"),
    device    = DEVICE,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
x = range(1, EPOCHS_P1 + EPOCHS_P2 + 1)

ax1.plot(x, hist_e2["train_loss"], label="Train")
ax1.plot(x, hist_e2["val_loss"],   label="Val")
ax1.axvline(EPOCHS_P1 + 0.5, color="gray", linestyle="--", alpha=0.5, label="Fase 1→2")
ax1.set_title("Exp 2 — Loss")
ax1.set_xlabel("Época")
ax1.legend()

ax2.plot(x, hist_e2["val_auc"], color="darkorange")
ax2.axvline(EPOCHS_P1 + 0.5, color="gray", linestyle="--", alpha=0.5)
ax2.set_title("Exp 2 — AUC-ROC Val")
ax2.set_xlabel("Época")
ax2.set_ylim(0, 1)

plt.suptitle("Experimento 2: U-Zeros (BCE estándar)", fontsize=12)
plt.tight_layout()
plt.show()

## (c) Experimento 3 — Label Smoothing ε = 0.55

**Inspirado en:** Yuan et al., "Large-scale Robust Deep AUC Maximization" (ICCV 2021).

**Política de incertidumbre:** los labels inciertos se asignan como **0.55** — un valor suave levemente por encima de 0.5.

**Justificación (Yuan et al.):** clínicamente, un hallazgo "incierto" (-1 en CheXpert) tiene mayor probabilidad de ser un verdadero positivo que un verdadero negativo. Asignar `ε = 0.55` en lugar de un hard label:
- Evita el sesgo fuerte de U-Ones (ε=1 introduce muchos falsos positivos)
- Reduce el impacto del ruido de etiquetado más que U-Zeros
- El valor 0.55 es el recomendado por el paper como punto de partida

El paper también propone una pérdida de margen AUC (AUCMLoss, disponible en la librería `libauc`) que combina este label smoothing con optimización directa del AUC. Para este experimento usamos BCE con smooth labels como aproximación.

In [ ]:
SMOOTH_EPS = 0.55  # valor propuesto por Yuan et al. para etiquetas inciertas

hist_e3, auc_e3, per_class_e3 = run_experiment(
    exp_name  = f"Exp 3 — Label Smoothing ε={SMOOTH_EPS}",
    nan_fill  = SMOOTH_EPS,   # NaN → 0.55
    criterion = standard_bce,
    save_path = os.path.join(DEV_DIR, "exp3_densenet121.pth"),
    device    = DEVICE,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
x = range(1, EPOCHS_P1 + EPOCHS_P2 + 1)

ax1.plot(x, hist_e3["train_loss"], label="Train")
ax1.plot(x, hist_e3["val_loss"],   label="Val")
ax1.axvline(EPOCHS_P1 + 0.5, color="gray", linestyle="--", alpha=0.5, label="Fase 1→2")
ax1.set_title("Exp 3 — Loss")
ax1.set_xlabel("Época")
ax1.legend()

ax2.plot(x, hist_e3["val_auc"], color="darkorange")
ax2.axvline(EPOCHS_P1 + 0.5, color="gray", linestyle="--", alpha=0.5)
ax2.set_title(f"Exp 3 — AUC-ROC Val")
ax2.set_xlabel("Época")
ax2.set_ylim(0, 1)

plt.suptitle(f"Experimento 3: Label Smoothing ε={SMOOTH_EPS}", fontsize=12)
plt.tight_layout()
plt.show()

## Tabla comparativa de experimentos

In [ ]:
exp_results = {
    "U-Ignore (Masked BCE)":        {"auc_val": auc_e1, "per_class": per_class_e1},
    "U-Zeros (BCE)":                {"auc_val": auc_e2, "per_class": per_class_e2},
    f"Label Smoothing ε={SMOOTH_EPS}": {"auc_val": auc_e3, "per_class": per_class_e3},
}

KEY_CLASSES = ["Atelectasis", "Cardiomegaly", "Edema", "Pleural_Effusion", "Pneumonia"]
key_idx = [CANONICAL_LABELS.index(c) for c in KEY_CLASSES]

rows = []
for exp_name, r in exp_results.items():
    row = {"Experimento": exp_name, "mean_AUC_val": f"{r['auc_val']:.4f}"}
    for c, i in zip(KEY_CLASSES, key_idx):
        v = r["per_class"][i]
        row[c] = f"{v:.4f}" if not np.isnan(v) else "—"
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index("Experimento")
print(comparison_df.to_string())

best_exp = max(exp_results, key=lambda k: exp_results[k]["auc_val"])
print(f"\nMejor experimento: {best_exp} (AUC val = {exp_results[best_exp]['auc_val']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = range(1, EPOCHS_P1 + EPOCHS_P2 + 1)

ax.plot(x, hist_e1["val_auc"], label="E1: U-Ignore",              marker="o", ms=3)
ax.plot(x, hist_e2["val_auc"], label="E2: U-Zeros",               marker="s", ms=3)
ax.plot(x, hist_e3["val_auc"], label=f"E3: Smooth ε={SMOOTH_EPS}", marker="^", ms=3)
ax.axvline(EPOCHS_P1 + 0.5, color="gray", linestyle="--", alpha=0.5, label="Fase 1→2")
ax.set_title("Comparación AUC-ROC en validación — 3 experimentos")
ax.set_xlabel("Época")
ax.set_ylabel("AUC-ROC")
ax.set_ylim(0.5, 1.0)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## (d) Evaluación final en test

Se evalúa el **mejor modelo** (mayor AUC en validación) sobre el conjunto de test.
El test set nunca se usó durante el entrenamiento ni la selección de hiperparámetros.

In [ ]:
# Determinar qué checkpoint cargar
best_paths = {
    "U-Ignore (Masked BCE)":        os.path.join(DEV_DIR, "exp1_densenet121.pth"),
    "U-Zeros (BCE)":                os.path.join(DEV_DIR, "exp2_densenet121.pth"),
    f"Label Smoothing ε={SMOOTH_EPS}": os.path.join(DEV_DIR, "exp3_densenet121.pth"),
}
best_nan_fills = {
    "U-Ignore (Masked BCE)":        None,
    "U-Zeros (BCE)":                0.0,
    f"Label Smoothing ε={SMOOTH_EPS}": SMOOTH_EPS,
}

best_model = build_densenet121().to(DEVICE)
best_model.load_state_dict(torch.load(best_paths[best_exp], map_location=DEVICE))
best_nan_fill = best_nan_fills[best_exp]

test_ds = ChestXRayDataset(
    os.path.join(DATA_DIR, "test.csv"), DATASET_ROOT,
    transform=val_test_transform, nan_fill=best_nan_fill
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type=="cuda")

print(f"Evaluando '{best_exp}' en test ({len(test_ds):,} imágenes)...")

In [ ]:
best_model.eval()
all_logits, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        logits = best_model(imgs)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

all_probs  = torch.cat(all_logits).sigmoid().numpy()
all_labels = torch.cat(all_labels).numpy()
all_preds  = (all_probs >= 0.5).astype(int)

# Métricas por clase
rows = []
for i, label in enumerate(CANONICAL_LABELS):
    col = all_labels[:, i]
    valid = ~np.isnan(col)
    if valid.sum() == 0 or len(np.unique(col[valid])) < 2:
        rows.append({"Clase": label, "AUC": "—", "F1": "—", "Precision": "—", "Recall": "—"})
        continue
    y_true = col[valid].astype(int)
    y_prob  = all_probs[valid, i]
    y_pred  = all_preds[valid, i]
    rows.append({
        "Clase":     label,
        "AUC":       f"{roc_auc_score(y_true, y_prob):.4f}",
        "F1":        f"{f1_score(y_true, y_pred, zero_division=0):.4f}",
        "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall":    f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
    })

metrics_df = pd.DataFrame(rows).set_index("Clase")
print("\n=== Métricas en TEST ===")
print(metrics_df.to_string())

In [ ]:
n_classes = NUM_CLASSES
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flat
fig.suptitle("Matrices de confusión por clase (test set, umbral 0.5)", fontsize=12)

for i, (label, ax) in enumerate(zip(CANONICAL_LABELS, axes)):
    col = all_labels[:, i]
    valid = ~np.isnan(col)
    if valid.sum() == 0:
        ax.axis("off")
        continue
    y_true = col[valid].astype(int)
    y_pred  = all_preds[valid, i]
    cm = confusion_matrix(y_true, y_pred)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(label, fontsize=8)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Neg", "Pos"], fontsize=7)
    ax.set_yticklabels(["Neg", "Pos"], fontsize=7)
    for r in range(cm.shape[0]):
        for c in range(cm.shape[1]):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", fontsize=7,
                    color="white" if cm[r, c] > cm.max()/2 else "black")

for ax in list(axes)[NUM_CLASSES:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Análisis de errores

Se visualizan las 8 imágenes donde el modelo tuvo mayor error: la diferencia entre probabilidad predicha y label real es máxima.

In [ ]:
IMAGENET_MEAN_T = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
IMAGENET_STD_T  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

# Cargar imágenes originales del test set para visualización (sin augmentation, con normalize)
test_ds_vis = ChestXRayDataset(
    os.path.join(DATA_DIR, "test.csv"), DATASET_ROOT,
    transform=val_test_transform, nan_fill=best_nan_fill
)

# Calcular error por muestra: MAE entre prob predicha y label (ignorando NaN)
test_csv = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
errors = []
for i in range(len(all_labels)):
    col = all_labels[i]
    valid = ~np.isnan(col)
    if valid.sum() == 0:
        errors.append(0.0)
    else:
        errors.append(np.abs(all_probs[i][valid] - col[valid]).mean())

worst_idx = np.argsort(errors)[-8:][::-1]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Análisis de errores — 8 predicciones con mayor error (test set)", fontsize=11)

for ax, idx in zip(axes.flat, worst_idx):
    img_tensor, _ = test_ds_vis[idx]
    img_np = (img_tensor * IMAGENET_STD_T + IMAGENET_MEAN_T).clamp(0, 1)
    img_np = img_np.permute(1, 2, 0).numpy().mean(axis=2)

    ax.imshow(img_np, cmap="gray")

    true_labels  = [CANONICAL_LABELS[j] for j in range(NUM_CLASSES)
                    if not np.isnan(all_labels[idx, j]) and all_labels[idx, j] == 1.0]
    pred_labels  = [CANONICAL_LABELS[j] for j in range(NUM_CLASSES)
                    if all_probs[idx, j] >= 0.5]

    title = f"Real: {', '.join(true_labels) or 'No Finding'}\nPred: {', '.join(pred_labels) or 'No Finding'}"
    ax.set_title(title, fontsize=6)
    ax.axis("off")

plt.tight_layout()
plt.show()

## (e) Guardado del modelo

Los pesos del mejor modelo se guardan en `dev/modelo.pth`.

### Git LFS
Los archivos `.pth` suelen superar los 100 MB. Para versionarlos en GitHub:
```bash
git lfs install
git lfs track "*.pth"
git add .gitattributes
git add dev/modelo.pth
git commit -m "add trained model weights"
```

In [ ]:
save_final = os.path.join(DEV_DIR, "modelo.pth")
torch.save(best_model.state_dict(), save_final)
size_mb = os.path.getsize(save_final) / (1024**2)
print(f"Modelo guardado: {save_final}")
print(f"Tamaño: {size_mb:.1f} MB")

# Verificar que se puede recargar
model_reload = build_densenet121()
model_reload.load_state_dict(torch.load(save_final, map_location="cpu"))
model_reload.eval()
print("Recarga verificada correctamente.")